# U-Net Baseline (Release)\n
\n
This notebook provides a clean release baseline for binary 2D segmentation using U-Net.\n
It supports loading real metadata-driven data and automatically falls back to a synthetic dataset for smoke testing.

In [1]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from dataset import SegmentationDataset, load_from_metadata, synthetic_blobs
from model import UNet
from train_eval import evaluate_unet, fit_unet

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [2]:
# Real metadata source from the baseline workflow.
metadata_csv = Path('/home/feity/compare/deepict/DeePiCt/2d_cnn/metas/nucleus_mitochondria_metadata.csv')

# This metadata uses columns: data (input image path), labels (mask path).
image_col = 'data'
mask_col = 'labels'
class_col = None
filter_class = None

# Use 512x512 and keep all slices whose label mask has positive pixels.
images, masks, slice_records = load_from_metadata(
    metadata_csv=metadata_csv,
    image_col=image_col,
    mask_col=mask_col,
    class_col=class_col,
    filter_class=filter_class,
    target_shape=(512, 512),
    return_slice_records=True,
)
data_source = f'real metadata: {metadata_csv}'

print('Data source:', data_source)
print('Images shape:', images.shape, 'Masks shape:', masks.shape)
print('Labeled slices selected:', len(slice_records))

Data source: real metadata: /home/feity/compare/deepict/DeePiCt/2d_cnn/metas/nucleus_mitochondria_metadata.csv
Images shape: (1386, 512, 512) Masks shape: (1386, 512, 512)
Labeled slices selected: 1386


In [3]:
# Check which slices were selected (only those with positive labels)
display(slice_records.head(20))

summary = (
    slice_records.groupby('mask_path', as_index=False)
    .agg(num_slices=('slice_index', 'count'), min_slice=('slice_index', 'min'), max_slice=('slice_index', 'max'))
    .sort_values('num_slices', ascending=False)
)

display(summary)
print('Total selected slices:', int(summary['num_slices'].sum()))

,image_path,mask_path,slice_index,positive_pixels
0,/data/transformer_project/transforemer_model/t...,/data/transformer_project/transforemer_model/t...,150,17161
1,/data/transformer_project/transforemer_model/t...,/data/transformer_project/transforemer_model/t...,151,17161
2,/data/transformer_project/transforemer_model/t...,/data/transformer_project/transforemer_model/t...,152,17161
3,/data/transformer_project/transforemer_model/t...,/data/transformer_project/transforemer_model/t...,153,15244
4,/data/transformer_project/transforemer_model/t...,/data/transformer_project/transforemer_model/t...,154,15244
5,/data/transformer_project/transforemer_model/t...,/data/transformer_project/transforemer_model/t...,155,15244
6,/data/transformer_project/transforemer_model/t...,/data/transformer_project/transforemer_model/t...,156,15244
7,/data/transformer_project/transforemer_model/t...,/data/transformer_project/transforemer_model/t...,157,15244
8,/data/transformer_project/transforemer_model/t...,/data/transformer_project/transforemer_model/t...,158,19436
9,/data/transformer_project/transforemer_model/t...,/data/transformer_project/transforemer_model/t...,159,19436


,mask_path,num_slices,min_slice,max_slice
5,/data/transformer_project/transforemer_model/t...,301,100,400
3,/data/transformer_project/transforemer_model/t...,301,100,400
4,/data/transformer_project/transforemer_model/t...,251,150,400
1,/data/transformer_project/transforemer_model/t...,191,140,330
0,/data/transformer_project/transforemer_model/t...,171,140,310
2,/data/transformer_project/transforemer_model/t...,171,190,360


Total selected slices: 1386


In [4]:
# Split with safeguards for very small datasets
n = len(images)
idx = np.arange(n)
rng = np.random.default_rng(SEED)
rng.shuffle(idx)

if n < 3:
    raise ValueError(f'Need at least 3 samples, found {n}.')

n_val = max(1, int(round(0.15 * n)))
n_test = max(1, int(round(0.15 * n)))
n_train = n - n_val - n_test

if n_train < 1:
    n_train = 1
    if n_val > n_test:
        n_val -= 1
    else:
        n_test -= 1

train_idx = idx[:n_train]
val_idx = idx[n_train:n_train + n_val]
test_idx = idx[n_train + n_val:n_train + n_val + n_test]

train_ds = SegmentationDataset(images[train_idx], masks[train_idx], augment=True)
val_ds = SegmentationDataset(images[val_idx], masks[val_idx], augment=False)
test_ds = SegmentationDataset(images[test_idx], masks[test_idx], augment=False)

# Keep batch size small for 512x512 inputs to reduce GPU memory pressure.
train_loader = DataLoader(train_ds, batch_size=min(2, len(train_ds)), shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=min(2, len(val_ds)), shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=min(2, len(test_ds)), shuffle=False, num_workers=0)

print('Train/Val/Test sizes:', len(train_ds), len(val_ds), len(test_ds))

Train/Val/Test sizes: 970 208 208


In [11]:
model = UNet(in_channels=1, out_channels=1, base_channels=32).to(device)
print('Parameters:', sum(p.numel() for p in model.parameters()))

out_dir = Path('./artifacts')
out_dir.mkdir(exist_ok=True)
ckpt_path = out_dir / 'best_unet_baseline.pth'

history = fit_unet(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=5,
    lr=1e-3,
    ckpt_path=ckpt_path,
)

Parameters: 1926433
Epoch 01 | train_loss=0.4766, train_iou=0.0839 | val_loss=0.6189, val_iou=0.0000
  Saved new best checkpoint: artifacts/best_unet_baseline.pth
Epoch 02 | train_loss=0.4160, train_iou=0.2123 | val_loss=0.7146, val_iou=0.1086
Epoch 03 | train_loss=0.3977, train_iou=0.2774 | val_loss=0.9459, val_iou=0.2079
Epoch 04 | train_loss=0.3746, train_iou=0.3483 | val_loss=0.9594, val_iou=0.1771
Epoch 05 | train_loss=0.3638, train_iou=0.3797 | val_loss=0.9926, val_iou=0.2078
